In [15]:
import pandas as pd
import re
from pathlib import Path

In [16]:
input_file = Path(r"D:\Proyek FEB\minggu 1\data_full_sqopus_sitasi.xlsx")  # ganti kalau file utamamu beda

df = pd.read_excel(input_file)

COL_JOURNAL = "Journal"
COL_Q = "SCOPUS Q"

In [17]:
def journal_key(value):
    if pd.isna(value):
        return ""
    
    text = str(value).lower().strip()
    text = re.sub(r"\s+", " ", text)
    
    # samakan simbol umum
    text = text.replace("&", "and")
    text = text.replace("–", "-").replace("—", "-")
    
    # rapikan spasi sekitar tanda baca
    text = re.sub(r"\s*-\s*", "-", text)
    text = re.sub(r"\s*:\s*", ": ", text)
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

In [18]:
q_conflict_override_raw = {
    "AIP Conference Proceedings": "no-Q",
    "Buku": "no-Q",
    "Book": "no-Q",
    "Books": "no-Q",
    "Banks and Bank Systems": "Q2",
    "Cogent Business and Management": "Q2",
    "Cogent Business & Management": "Q2",
    "Discover Sustainability": "Q1",
    "International Journal of Data and Network Science": "Q1",
    "International Journal of Islamic and Middle Eastern Finance and Management": "Q2",
    "Journal of Islamic Accounting and Business Research": "Q2",
    "Journal of Islamic Marketing": "Q2",
    "Journal of Risk and Financial Management": "Q2",
    "Compensation and Benefits Review": "Q2",
    "Compensation & Benefits Review": "Q2",
}

q_conflict_override = {
    journal_key(journal): q
    for journal, q in q_conflict_override_raw.items()
}

q_conflict_override

{'aip conference proceedings': 'no-Q',
 'buku': 'no-Q',
 'book': 'no-Q',
 'books': 'no-Q',
 'banks and bank systems': 'Q2',
 'cogent business and management': 'Q2',
 'discover sustainability': 'Q1',
 'international journal of data and network science': 'Q1',
 'international journal of islamic and middle eastern finance and management': 'Q2',
 'journal of islamic accounting and business research': 'Q2',
 'journal of islamic marketing': 'Q2',
 'journal of risk and financial management': 'Q2',
 'compensation and benefits review': 'Q2'}

In [19]:
df_fixed = df.copy()
df_fixed["journal_key"] = df_fixed[COL_JOURNAL].apply(journal_key)

override_count = 0

for idx, row in df_fixed.iterrows():
    key = row["journal_key"]
    
    if key in q_conflict_override:
        new_q = q_conflict_override[key]
        old_q = df_fixed.at[idx, COL_Q]
        
        if str(old_q).strip() != str(new_q).strip():
            df_fixed.at[idx, COL_Q] = new_q
            override_count += 1

print("Jumlah baris yang diperbaiki/diseragamkan:", override_count)

Jumlah baris yang diperbaiki/diseragamkan: 28


In [20]:
check_keys = set(q_conflict_override.values())

df_fixed[
    df_fixed["journal_key"].isin(q_conflict_override.keys())
][[COL_JOURNAL, COL_Q]].drop_duplicates().sort_values(COL_JOURNAL)

,Journal,SCOPUS Q
430,AIP Conference Proceedings,no-Q
2205,Aip Conference Proceedings,no-Q
1020,Banks and Bank Systems,Q2
590,Buku,no-Q
163,Cogent Business & Management,Q2
611,Cogent Business and Management,Q2
1932,Compensation & Benefits Review,Q2
2192,Compensation and Benefits Review,Q2
1767,Discover Sustainability,Q1
810,International Journal of Data and Network Science,Q1


In [21]:
def is_empty_value(value):
    if pd.isna(value):
        return True
    
    text = str(value).strip()
    
    return text == "" or text.lower() in [
        "nan", "none", "null", "-", "?",
        "??", "???", "n/a", "na",
        "tidak ada", "not found"
    ]


def clean_q_label(value):
    if is_empty_value(value):
        return "EMPTY"
    
    text = str(value).strip()
    
    if re.search(r"\bno-?q\b", text, flags=re.IGNORECASE):
        return "no-Q"
    
    match = re.search(r"\bq\s*([1-4])\b", text, flags=re.IGNORECASE)
    if match:
        return f"Q{match.group(1)}"
    
    return text

In [22]:
df_fixed["q_clean"] = df_fixed[COL_Q].apply(clean_q_label)

df_non_empty_q = df_fixed[df_fixed["q_clean"] != "EMPTY"].copy()

conflict_summary_after = (
    df_non_empty_q
    .groupby("journal_key")
    .agg(
        rows=("journal_key", "size"),
        journal_name=(COL_JOURNAL, lambda x: x.dropna().astype(str).iloc[0] if len(x.dropna()) > 0 else ""),
        q_unique_count=("q_clean", "nunique"),
        q_values=("q_clean", lambda x: " | ".join(sorted(set(x))))
    )
    .reset_index()
)

conflict_summary_after = conflict_summary_after[
    conflict_summary_after["q_unique_count"] > 1
].copy()

print("Jumlah konflik Q setelah override:", len(conflict_summary_after))

conflict_summary_after.sort_values(["q_unique_count", "rows"], ascending=[False, False]).head(50)

Jumlah konflik Q setelah override: 0


,journal_key,rows,journal_name,q_unique_count,q_values


In [23]:
COL_JOURNAL = "Journal"

In [24]:
manual_journal_q_by_no = {
    410: {
        "Journal": "Proceedings of the 2020 11th International Conference on E-Education, E-Business, E-Management, and E-Learning",
        "SCOPUS Q": "no-Q"
    },
    900: {
        "Journal": "The Emerald Handbook of Destination Recovery in Tourism and Hospitality",
        "SCOPUS Q": "no-Q"
    },
    915: {
        "Journal": "Social Morphology, Human Welfare, and Sustainability",
        "SCOPUS Q": "no-Q"
    },
    916: {
        "Journal": "Handbook of Research on Global Networking Post COVID-19",
        "SCOPUS Q": "no-Q"
    },
    946: {
        "Journal": "Self-Care and Stress Management for Academic Well-Being",
        "SCOPUS Q": "no-Q"
    },
    1833: {
        "Journal": "Islamic Finance in the Digital Age",
        "SCOPUS Q": "no-Q"
    },
    1834: {
        "Journal": "Islamic Finance in the Digital Age",
        "SCOPUS Q": "no-Q"
    },
    1835: {
        "Journal": "Islamic Finance in the Digital Age",
        "SCOPUS Q": "no-Q"
    },
}

In [25]:
COL_NO = "No"
COL_JOURNAL = "Journal"
COL_Q = "SCOPUS Q"

updated_count = 0

for no, values in manual_journal_q_by_no.items():
    mask = df_fixed[COL_NO] == no
    
    if mask.sum() == 0:
        print(f"No {no} tidak ditemukan")
        continue
    
    df_fixed.loc[mask, COL_JOURNAL] = values["Journal"]
    df_fixed.loc[mask, COL_Q] = values["SCOPUS Q"]
    
    updated_count += mask.sum()

print("Jumlah baris yang diperbarui:", updated_count)

Jumlah baris yang diperbarui: 8


In [26]:
df_fixed[df_fixed[COL_NO].isin(manual_journal_q_by_no.keys())][
    ["No", "Title", "Authors", "Journal", "SCOPUS Q"]
]

,No,Title,Authors,Journal,SCOPUS Q
409,410,Customer E-Loyalty of Muslim Millennials in In...,SRI HERIANINGRUM,Proceedings of the 2020 11th International Con...,no-Q
899,900,Strategic Intent and Strategic Leadership: A R...,DIAN EKOWATI,The Emerald Handbook of Destination Recovery i...,no-Q
914,915,Social perspective: Leadership in changing soc...,DIAN EKOWATI,"Social Morphology, Human Welfare, and Sustaina...",no-Q
915,916,Integrating cycle of prochaska and diclemente ...,DIAN EKOWATI,Handbook of Research on Global Networking Post...,no-Q
945,946,Human capital development in youth inspires us...,DIAN EKOWATI,Self-Care and Stress Management for Academic W...,no-Q
1832,1833,A critical appraisal of the Sovereign Green Is...,DIAN AGUSTIA,Islamic Finance in the Digital Age,no-Q
1833,1834,A critical appraisal of the Sovereign Green Is...,RADITYA SUKMANA,Islamic Finance in the Digital Age,no-Q
1834,1835,A critical appraisal of the Sovereign Green Is...,NISFUL LAILA,Islamic Finance in the Digital Age,no-Q


In [27]:
df_fixed.isnull().sum()

No                    0
Title                 0
Authors               0
Journal               0
SCOPUS Q              0
Tahun                 1
Volume / Issue      108
LINK DOI/ARTIKEL      0
SCOPUS SITASI         0
journal_key           0
q_clean               0
dtype: int64

In [28]:
df_fixed.drop(columns=["journal_key", "q_clean"], inplace=True)

In [29]:
df_fixed.to_excel("data_full_sqopus_sitasi_fixed.xlsx", index=False)